### Bibliotecas necessárias.

In [1]:
import json
import re
import time
import pandas as pd
from pathlib import Path
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

### Autenticação no hugging face.

In [2]:
class OABAuth:
    """Responsável por autenticação em serviços externos."""
    @staticmethod
    def login_hugging_face(secret_name='hugging_colab'):
        try:
            token = userdata.get(secret_name)
            login(token=token)
            print("Login no Hugging Face realizado com sucesso.")
        except Exception as e:
            print(f"Erro na autenticação: {e}")

### Classificar questões.

In [3]:
class OABClassifier:
    """Responsável por métricas de complexidade jurídica."""

    @staticmethod
    def count_laws(text: str) -> int:
        if not text: return 0
        # Padrões jurídicos comuns em questões da OAB
        patterns = [
            r'\blei\b', r'\bart\.?\b', r'\bartigo\b',
            r'\bdecreto\b', r'\bconstituição\b', r'\bcf/88\b', r'\bsúmula\b'
        ]
        combined_pattern = '|'.join(patterns)
        return len(re.findall(combined_pattern, str(text).lower()))

    @classmethod
    def apply_difficulty(cls, df: pd.DataFrame) -> pd.DataFrame:
        """Define o nível analisando o enunciado e alternativas (pois não há explicação)."""
        def get_level(row):
            # Analisa o texto da questão + texto das alternativas
            full_text = f"{row.get('question', '')} {row.get('choices', '')}"
            n = cls.count_laws(full_text)

            if n == 0: return 1 # Estagiário
            if n <= 2: return 2 # Analista
            if n <= 4: return 3 # Juiz
            return 4           # Ministro

        df['level'] = df.apply(get_level, axis=1)
        return df

### Carregar questões e linha guia.

In [4]:
class OABDataProcessor:
    """Gerencia a carga, tradução e limpeza dos dados do Hugging Face."""

    CATEGORIES_MAP = {
        'ETHICS': 'Ética', 'INTERNATIONAL': 'Internacional', 'CONSTITUTIONAL': 'Constitucional',
        'BUSINESS': 'Negócios', 'CONSUMER': 'Consumidor', 'CIVIL': 'Civil',
        'CIVIL-PROCEDURE': 'Processo Civil', 'ADMINISTRATIVE': 'Administrativo',
        'TAXES': 'Tributário', 'LABOUR': 'Trabalho', 'LABOUR-PROCEDURE': 'Processo do Trabalho',
        'ENVIRONMENTAL': 'Ambiental', 'CRIMINAL': 'Penal',
        'CRIMINAL-PROCEDURE': 'Processo Penal', 'CHILDREN': 'ECA',
        'HUMAN-RIGHTS': 'Direitos Humanos', 'PHILOSOPHY': 'Filosofia', 'PHILOSOPY': 'Filosofia'
    }

    def __init__(self, dataset_id='eduagarcia/oab_exams'):
        self.dataset_id = dataset_id
        self.classifier = OABClassifier()

    def load_clean_data(self) -> pd.DataFrame:
        """Carrega o dataset e aplica todas as transformações de POO."""
        print(f"A carregar dataset: {self.dataset_id}...")
        ds = load_dataset(self.dataset_id)
        df = pd.DataFrame(ds['train'])

        # 1. Tradução de categorias
        df['category'] = df['question_type'].map(self.CATEGORIES_MAP).fillna(df['question_type'])

        # 2. Padronização de colunas
        df = df.rename(columns={'exam_id': 'edition', 'exam_year': 'year', 'answerKey': 'answer'})

        # 3. Adição de metadados fixos
        df.insert(0, 'num', range(1, len(df) + 1))
        df['type'] = 'Fechada'

        # 4. Aplicação da dificuldade (Nível)
        df = self.classifier.apply_difficulty(df)

        return df[['num', 'edition', 'year', 'category', 'question', 'choices', 'answer', 'level', 'type']]

### Classe principal.

In [5]:
class OABManager:
    """Classe principal para coordenar a execução do pipeline."""

    @staticmethod
    def setup_auth():
        try:
            token = userdata.get('hugging_colab')
            login(token=token)
        except Exception as e:
            print(f"Aviso de Autenticação: {e}")

    @staticmethod
    def save_results(df, filename='close_questions_final.jsonl'):
        df.to_json(filename, orient='records', lines=True, force_ascii=False)
        print(f"Sucesso! Arquivo guardado como: {filename}")

### Execução

In [6]:
# 1. Setup
manager = OABManager()
manager.setup_auth()

# 2. Processamento
processor = OABDataProcessor()
df_questions = processor.load_clean_data()

# 3. Exemplo de filtro (Questões 319 a 424)
# Usando a lógica de valor que discutimos anteriormente
df_my_questions = df_questions[(df_questions['num'] >= 319) & (df_questions['num'] <= 424)].copy()

# 4. Guardar resultados
manager.save_results(df_questions, 'close_questions.jsonl')

A carregar dataset: eduagarcia/oab_exams...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/649 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2210 [00:00<?, ? examples/s]

Sucesso! Arquivo guardado como: close_questions.jsonl
